In [ ]:
import pandas as pd
import statsmodels.api as sm

# Load data
df = pd.read_csv("../signals.csv")

# Convert date and sort chronologically
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Define predictors and target
signal_cols = [
    "1d_return",
    "5d_return",
    "20d_return",
    "percent_ma_gap",
    "sma_crossover",
    "20d_drawdown",
    "10d_volatility",
    "volatility_ratio",
    "volume_zscore",
    "rsi_14"
]

target = "target"

# Keep only relevant columns
model_df = df[["date"] + signal_cols + [target]].copy()

# Remove rows with missing signal/target values
model_df = model_df.dropna(
    subset=signal_cols + [target]
).reset_index(drop=True)

print(model_df.shape)
print(model_df.columns)

(1235, 12)
Index(['date', '1d_return', '5d_return', '20d_return', 'percent_ma_gap',
       'sma_crossover', '20d_drawdown', '10d_volatility', 'volatility_ratio',
       'volume_zscore', 'rsi_14', 'target'],
      dtype='object')


In [ ]:
import os
import pandas as pd
import statsmodels.api as sm

# 1. Create output folders
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# 2. Chronological Split (80% Train/Dev, 20% Holdout Test)
split_idx = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

# 3. Fit Statsmodels Logistic Regression on Training Data ONLY
X_train = train_df[signal_cols]
X_train_const = sm.add_constant(X_train)  # Includes intercept
y_train = train_df[target]

logit_model = sm.Logit(y_train, X_train_const).fit()

# Print summary to verify model fit
print(logit_model.summary())

# 4. Save results/regression_coefficients.csv
coeff_df = pd.DataFrame({
    "feature": logit_model.params.index,
    "coefficient": logit_model.params.values,
    "std_err": logit_model.bse.values,
    "z_stat": logit_model.tvalues.values,
    "p_value": logit_model.pvalues.values,
    "conf_lower": logit_model.conf_int()[0].values,
    "conf_upper": logit_model.conf_int()[1].values,
})
coeff_df.to_csv("results/regression_coefficients.csv", index=False)

# 5. Save results/model_predictions.csv
X_full = model_df[signal_cols]
X_full_const = sm.add_constant(X_full)

predictions_df = model_df.copy()
predictions_df["predicted_prob"] = logit_model.predict(X_full_const)
predictions_df["sample_split"] = "train_dev"
predictions_df.loc[split_idx:, "sample_split"] = "test_holdout"

predictions_df.to_csv("results/model_predictions.csv", index=False)

# 6. Generate docs/model_notes.md
doc_text = f"""# Baseline Logistic Regression Model Notes

## Model Summary
{logit_model.summary().as_text()}

## Key Findings
- The model is a logistic regression trained on the specified signals to predict the `target` variable.
- The data was chronologically split into 80% train/dev and 20% holdout test sets.
- Coefficients for each feature are available in `results/regression_coefficients.csv`.
- Predictions for the full dataset are saved in `results/model_predictions.csv`.
"""

with open("docs/model_notes.md", "w") as f:
    f.write(doc_text)

Optimization terminated successfully.
         Current function value: 0.675411
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:                  988
Model:                          Logit   Df Residuals:                      977
Method:                           MLE   Df Model:                           10
Date:                Thu, 13 Aug 2026   Pseudo R-squ.:                0.006228
Time:                        17:07:32   Log-Likelihood:                -667.31
converged:                       True   LL-Null:                       -671.49
Covariance Type:            nonrobust   LLR p-value:                    0.5934
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                1.3020      0.661      1.970      0.049       0.006       2.598
1d_return  